In [1]:
import torch
import torch.nn as nn
from torchvision.transforms import Compose, ToTensor, Normalize
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.datasets import CIFAR10
import torch.nn.functional as F
from torch.utils.data import DataLoader

from utils import get_optimal_device, RandomDataset
from utils import plot_cifar_like_tensor, denorm

In [2]:
device = get_optimal_device()
print(f'Using device: __{device}__')


Using device: __mps__


In [4]:
# CIFAR 10
mean = (1,1,1)
std = (2,2,2)
tf = Compose([ToTensor(), Normalize(mean, std)])
cifar10 = DataLoader(CIFAR10("./", train=True, transform=tf,
                     download=True), batch_size=128, shuffle=True)
x, y_true = next(iter(cifar10))
plot_cifar_like_tensor(denorm(x[0], mean ,std))

NameError: name 'denorm' is not defined

In [14]:
train_dataset = RandomDataset(1000,3,32,10)
dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
x, y_true = next(iter(dataloader))

In [89]:
# Example of target with class indices
torch.manual_seed(1137)
loss = nn.CrossEntropyLoss()
input = torch.randn(3, 5, requires_grad=True)
target = torch.empty(3, dtype=torch.long).random_(5)
print(target)
output = loss(input, target)
output.backward()
output

tensor([1, 3, 4])


tensor(1.1935, grad_fn=<NllLossBackward0>)

In [100]:
# Example of target with class probabilities
torch.manual_seed(1137)
loss1 = nn.CrossEntropyLoss()
loss2 = nn.NLLLoss()
loss3 = nn.CrossEntropyLoss()

# raw logits
logits = torch.randn(5, 10)
probs = logits.softmax(dim=1)
labels = torch.tensor([9, 3, 2, 8, 3], dtype=torch.long)
labels_one_hot = F.one_hot(labels, 10).float()

output1 = loss1(logits, labels)
output2 = loss2(logits.log_softmax(dim=1), labels)
output3 = loss3(logits, labels_one_hot)
print(f'CE Loss: {output1}')
print(f'NLL Loss: {output2}')
print(f'CE_probs Loss: {output3}')

CE Loss: 1.1035596132278442
NLL Loss: 1.1035596132278442
CE_probs Loss: 1.1035596132278442


In [91]:
torch.manual_seed(1137)
loss = nn.NLLLoss()
input= torch.randn(1,10,requires_grad=True)
target = torch.randn(1,10).log_softmax(dim=1)
output = loss(input, target.argmax(dim=1))
output.backward()
output


tensor(0.6286, grad_fn=<NllLossBackward0>)